# Regime Model Pipeline Orchestration

Thin runner for the Phase 2 pipeline modules.

`config.use_existing_intermediate = False` now rebuilds the reviewed notebook-1 dataset stage from raw inputs before partition export.

## 1. Setup / Imports

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from models.regime_model import RunConfig, build_feature_dataset, train_regime_xgb, run_full_pipeline

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None) # Show all content of each column
pd.set_option('display.width', 1000)        # Set the display width to 1000 characters
pd.options.display.float_format = '{:,.5f}'.format


## 2. Run Configuration

Set `use_existing_intermediate=False` to rebuild the dataset stage from raw inputs. Leave it as `True` to load the reviewed intermediate partitions directly.

In [ ]:
config = RunConfig(
    round_name='round_1',
    long_short=-1,
    debug_rows=None,
    use_existing_intermediate=False,
    save_intermediate=True,
    save_model=True,
    save_tables=True,
    save_figures=False,
    overwrite_outputs=True,
    use_selected_features=True,
)

config

## 3. Stage Toggles

In [ ]:
RUN_DATASET_STAGE = True
RUN_TRAIN_STAGE = False
RUN_FULL_PIPELINE = False

## 4. Run Selected Stage(s)

In [ ]:
dataset_result = None
training_result = None
full_result = None

if RUN_FULL_PIPELINE:
    full_result = run_full_pipeline(config)
    dataset_result = full_result['dataset']
    training_result = full_result['training']
else:
    if RUN_DATASET_STAGE:
        dataset_result = build_feature_dataset(config)
    if RUN_TRAIN_STAGE:
        training_result = train_regime_xgb(config, dataset=dataset_result)

{
    'dataset_loaded': dataset_result is not None,
    'training_run': training_result is not None,
    'full_pipeline_run': full_result is not None,
}

## 5. Lightweight Summary / Inspection

The notebook stays intentionally thin: config, toggles, pipeline call, and compact result inspection only.

In [ ]:
if dataset_result is not None:
    print('Dataset stage source:', dataset_result.source)
    print('Row counts:', dataset_result.row_counts)
    print('Column counts:', dataset_result.column_counts)
    print('Date ranges:', dataset_result.date_ranges)

if training_result is not None:
    print('Best params:', training_result.best_params)
    print('Selected feature count:', len(training_result.selected_feature_columns))
    print('RFECV feature count:', len(training_result.rfecv_feature_columns))
    print('Artifact paths:')
    for name, path in training_result.artifact_paths.items():
        print(f'  {name}: {path}')

In [ ]:
from inspect import getsource
print(getsource(build_feature_dataset))

In [ ]:
# p = r"C:\Users\mateo\OneDrive\Documents\algo-codex\data\intermediate\regime_model\round_1\feature_dataset_full.parquet"
# df_chk = pd.read_parquet(p)
# print(df_chk.shape)

# config = RunConfig(
#     use_existing_intermediate=True,
#     debug_rows=None,   # irrelevant for this path
# )
# dataset = build_feature_dataset(config)
# {k: v.shape for k, v in dataset.partitions.items()}

## 6. Testing Training Pipeline with Save Data

In [ ]:
config = RunConfig(
    round_name='round_1',
    long_short=-1,
    debug_rows=35000,
    use_existing_intermediate=True,
    save_intermediate=True,
    save_model=True,
    save_tables=True,
    save_figures=True,
    overwrite_outputs=True,
    use_selected_features=True,
    classification_threshold = 0.35
)

config

In [ ]:
RUN_DATASET_STAGE = False
RUN_TRAIN_STAGE = True
RUN_FULL_PIPELINE = False

In [ ]:
dataset_result = None
training_result = None
full_result = None

if RUN_FULL_PIPELINE:
    full_result = run_full_pipeline(config)
    dataset_result = full_result['dataset']
    training_result = full_result['training']
else:
    if RUN_DATASET_STAGE:
        dataset_result = build_feature_dataset(config)
    if RUN_TRAIN_STAGE:
        training_result = train_regime_xgb(config, dataset=dataset_result)

{
    'dataset_loaded': dataset_result is not None,
    'training_run': training_result is not None,
    'full_pipeline_run': full_result is not None,
}

In [ ]:
if training_result is not None:
    print('Best params:', training_result.best_params)
    print('Selected feature count:', len(training_result.selected_feature_columns))
    print('RFECV feature count:', len(training_result.rfecv_feature_columns))
    print('Artifact paths:')
    for name, path in training_result.artifact_paths.items():
        print(f'  {name}: {path}')